In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-5-mini", temperature=0)
response = llm.invoke("Hola")
response.text

'Hola. ¿En qué puedo ayudarte hoy?'

In [ ]:
system_prompt = """\
Eres un asistente de ventas que ayuda a los clientes a encontrar los productos que 
necesitan.

Los productos de nuestra tienda son:
- Laptop
- Mouse
- Teclado
- Audifonos
"""

messages = [
    ("system", system_prompt),
    ("user", "Dime que productos ofrecen en la tienda y si hay un teclado")
]

response = llm.invoke(messages)
response.text

'Ofrecemos estos productos: Laptop, Mouse, Teclado y Audífonos.  \n\nSí, tenemos teclado. ¿Te interesa ver modelos, precios o características?'

In [7]:
import requests
from langchain_core.tools import tool


In [23]:
@tool(
    "get_products",
    description="""Get all the products that the store offers""",
)
def get_products():
    products_response = requests.get("https://api.escuelajs.co/api/v1/products")
    products = products_response.json()
    return ", ".join([f"{p.get('title')} - {p.get('price')}" for p in products])


In [12]:
get_products.invoke({"price": 50})

'Majestic Mountain Graphic T-Shirt - 44, Chage title - 100, Classic Comfort Fit Joggers - 25, Classic Comfort Drawstring Joggers - 79, Classic Red Jogger Sweatpants - 98, Classic Navy Blue Baseball Cap - 61, Classic Blue Baseball Cap - 86, Classic Red Baseball Cap - 35, Classic Black Baseball Cap - 58, Classic Olive Chino Shorts - 84, Classic High-Waisted Athletic Shorts - 43, Classic White Crew Neck T-Shirt - 39, Classic White Tee - Timeless Style and Comfort - 73, Classic Black T-Shirt - 35, Updated Product - 250, Sleek Comfort-Fit Over-Ear Headphones - 28, Efficient 2-Slice Toaster - 48, Sleek Wireless Computer Mouse - 10, Sleek Modern Laptop with Ambient Lighting - 43, Sleek Modern Laptop for Professionals - 97, Stylish Red & Silver Over-Ear Headphones - 39, Sleek Mirror Finish Phone Case - 27, Sleek Smartwatch with Vibrant Display - 16, Sleek Modern Leather Sofa - 53, Mid-Century Modern Wooden Dining Table - 24, Elegant Golden-Base Stone Top Dining Table - 66, Modern Elegance Teal

In [10]:
@tool("get_weather", description="Get the weather of a city")
def get_weather(city: str):
    response_meteo = requests.get(
        f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1"
    )
    data = response_meteo.json()
    first_item = data.get("results", [])[0]
    latitude = first_item.get("latitude")
    longitude = first_item.get("longitude")
    response_forecast = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true"
    )
    forecast_data = response_forecast.json()
    temperature = forecast_data["current_weather"]["temperature"]
    windspeed = forecast_data["current_weather"]["windspeed"]

    return f"The wather in {city} is {temperature}C with {windspeed}"


get_weather.invoke({"city": "Cali"})


'The wather in Cali is 24.4C with 4.2'

In [24]:
system_prompt = """\
Eres un asistente de ventas que ayuda a los clientes a encontrar los productos que 
necesitan, ademas ayudaras a los clientes a dar el clima de la ciudad que soliciten.
"""

messages = [
    ("system", system_prompt),
    ("user", "Qué productos se ofrecen en la tienda"),
]

llm_with_tools = llm.bind_tools([get_products, get_weather])
llm_response = llm_with_tools.invoke(messages)
llm_response.tool_calls

[{'name': 'get_products',
  'args': {},
  'id': 'call_gCIy8KuF0ye61hL1ku75R0eI',
  'type': 'tool_call'}]